# 2. Inspect causal shape features
CoV uses linear power; CoV of dBm is not physically scale invariant.
Other features use daily-seasonality residuals standardised with training data.
CUSUM uses the strictly prior rolling mean. Fixed entropy bins preserve a common
meaning across windows. Derivatives are expressed per hour, not per row.

In [ ]:
from pathlib import Path
import sys
import json
import pandas as pd
import matplotlib.pyplot as plt

ROOT = Path.cwd()
if not (ROOT / "src").exists():
    ROOT = ROOT.parent
sys.path.insert(0, str(ROOT / "src"))
# Resolve relative configured output paths consistently from any notebook.
import os
os.chdir(ROOT)
from optical_anomaly.pipeline import prepare, develop, final_evaluation

CONFIG_PATH = ROOT / "configs/config.yaml"
RUN = prepare(CONFIG_PATH)
settings = json.loads((RUN / "settings.json").read_text())
start = pd.Timestamp("2025-01-01", tz="UTC")
boundaries = [start + pd.Timedelta(days=settings["generator"]["days"] * f)
              for f in settings["splits"]]


In [ ]:
from optical_anomaly.adapter import TelemetryAdapter
from optical_anomaly.validation import DataValidator
from optical_anomaly.features import FeatureEngineer, FEATURES

native = pd.read_parquet(RUN / "telemetry.parquet", filters=[("time", "<", boundaries[0])])
interval = settings["generator"]["interval_minutes"]
telemetry = DataValidator(f"{interval}min").transform(TelemetryAdapter().transform(native))
engineer = FeatureEngineer(interval_minutes=interval, **settings["features"]).fit(telemetry)
features = engineer.transform(telemetry)
display(features[FEATURES].describe())
features[FEATURES].hist(bins=30, figsize=(12, 7))
plt.tight_layout()
plt.show()
print("Complete feature coverage:", features[FEATURES].notna().all(axis=1).mean())


A missing reading restarts the rolling history. No imputation borrows future
values. Normalisation requires a representative local baseline; portability is not
zero-shot transfer. Unknown entities abstain. Seasonal correction is a modelling
assumption: compare `seasonal: false` in a new run before deciding it helps.
Entropy of constant/saturated bins can decrease during a fault; Isolation Forest
can use either direction. CUSUM's rolling reference may absorb very slow drift.